In [ ]:
include( "../geom.jl" )
include( "../cluster.jl" )
include( "../visual.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )

home, base = expfolder( "system" );

In [ ]:
# Hyper parameters.
N = 1024
r = 1.0

# Speed and density parameters.
ρ = 1.0
s = 0.251
L = √(N/ρ)

# Extract folder name and extract critical noise.
folder = createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) )
η = readdlm( folder*"../order-0.5.txt" )[1]

# Create parameter variable.
params = Dict{Char,Real}( 'N' => N, 'L' => L, 'r' => r, 'v' => s, 'η' => η )

# Print details.
println( "Particle speed:      $(round( s, digits=3 ))." )
println( "System-wide density: $(round( ρ, digits=3 ))." )
println( "Critical noise:      $(round( η, digits=3 ))." )

In [ ]:
# Initialize progress meter.
T = 10_000
meter = Progress( T, "Running to steady state."; dt=1.0 )

# Initial conditions.
x = L*rand( N, )
y = L*rand( N, )
θ = 2*π*rand( N, ) .- π

# Run simulation.
φlist = Vector{defFloat}( undef, T + 1 )
φlist[1] = order( N, θ )
for t ∈ 1:T
    A = adjacency( N, L, r, x, y, θ )
    x, y, θ = vicsek( params, A, x, y, θ )
    φlist[t+1] = order( N, θ )
    next!( meter )
end

In [ ]:
# Plot the order parameter over time.
plt = plot(; size=(400,200) )

plot!( plt, 0:T, φlist, color=:black, lw=2, label="" )

plot!( plt; xlims=(0,T) )
plot!( plt; ylims=(0,1) )

plot!( plt, xlabel=L"t", ylabel=L"\varphi", guidefont=10 )

In [ ]:
# Initialize steady state variables for saving.
Tsave = 100_000
φsave = Vector{defFloat}( undef, Tsave + 1 )

# Initial conditions.
φsave[1] = φlist[end];

# Initialize cluster information.
Â = Dict{defInt,Vector{defFloat}}( [i => Vector{defFloat}() for i ∈ 2:N] )
ℓ = Dict{defInt,Vector{defFloat}}( [i => Vector{defFloat}() for i ∈ 2:N] )

# Initialize cluster information.
ιp = clusterinfo( N, adjacency( N, L, r, x, y, θ ) )[2];

In [ ]:
# Initialize progress meter.
meter = Progress( Tsave, "Running simulation."; dt=1.0 )

# Run simulation from steady state.
for t ∈ 1:Tsave
    # Compute the current adjacency matrix.
    A = adjacency( N, L, r, x, y, θ )

    # Extract cluster information from adjacency.
    if (t - 1)%4 == 0
        countgeomlist!( L, r, Â, ℓ, x, y, A )
    end

    # Upadate the position of each agent.
    x, y, θ = vicsek( params, A, x, y, θ )

    # Update progress meter.
    next!( meter )
end

In [ ]:
# xℓlist = vec( [fill( ι, length( ℓ[ι] ) ) for ι in ιlist] )
# yℓlist = vec( [ℓ[ι] for ι in ιlist] )
# βℓ = fitpolylog( vcat( xℓlist... ), vcat( yℓlist... ) )

In [ ]:
# Compute mean and standard deviation.
ιmin = 2
ιlist = [ι for ι in ιmin:length( Â ) if (2 < length( Â[ι] ))]
ιmax = round( log10( N ) )

# Compute the area across the cluster range.
μAlist = vec( [mean( Â[ι] ) for ι in ιlist] )
σAlist = vec( [std(  Â[ι] ) for ι in ιlist] )

# Compute the perimeter across the cluster range.
μℓlist = vec( [mean( ℓ[ι] ) for ι in ιlist] )
σℓlist = vec( [std(  ℓ[ι] ) for ι in ιlist] )

xℓlist = vec( [fill( ι, length( ℓ[ι] ) ) for ι in ιlist] )
yℓlist = vec( [ℓ[ι] for ι in ιlist] )

# Perimeter.
xℓdata = [[vcat( [fill( ι, length( ℓdata[i][j][ι] ) ) for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ]
yℓdata = [[vcat( [ℓdata[i][j][ι] for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ];

# Extract poly-logarithmic parameters.
βA = fitpolylog( ιlist, μAlist )
βℓ = fitpolylog( ιlist, μℓlist )
println( "Poly-log trend:       ", βℓ, βA )

# Extract power law parameters.
λA = fitpower( ιlist[10:end], μAlist[10:end] )
λℓ = fitpower( ιlist[10:end], μℓlist[10:end] )
println( "Power-law trend:      ", λℓ, λA )

# Extract the log-power-law parameters.
γA = fitlogpower( ιlist, μAlist )
γℓ = fitlogpower( ιlist, μℓlist );
println( "Power-log-law trend:  ", γℓ, γA )

In [ ]:
# Plot the perimeter to cluster size trend.
plt = plot(; size=(300,200), legend=false )

scatter!( plt, ιlist, μAlist; yerror=σAlist, color=:gray )
# plot!( plt, 1:N, powerlaw.( 1:N; λ=λA ); color=:black, lw=2 )
plot!( plt, 1:N, polyloglaw.( 1:N; β=βA ); color=:black, lw=2 )
# plot!( plt, 1:N, logpowerlaw.( 1:N; γ=γA ); color=:black, lw=2 )

plot!( plt; xlims=(1,10^ιmax), xscale=:log10 )
plot!( plt; ylims=(0.01,1000), yscale=:log10 )

In [ ]:
# Plot the perimeter to cluster size trend.
plt = plot(; size=(300,200), legend=false )

scatter!( plt, ιlist, μℓlist; yerror=σℓlist, color=:gray )
# plot!( plt, 1:N, powerlaw.( 1:N; λ=λℓ ); color=:cornflowerblue, lw=2 )
plot!( plt, 1:N, polyloglaw.( 1:N; β=βℓ ); color=:cornflowerblue, lw=2 )
# plot!( plt, 1:N, logpowerlaw.( 1:N; γ=γℓ ); color=:indianred, lw=2 )

plot!( plt; xlims=(1,N), xticks=(xticks, string.( round.( defInt, xticks ) )), xscale=:log10 )
plot!( plt; ylims=(0.01,100), yscale=:log10 )
plot!( plt; framestyle=:box )

In [ ]:
Âmax = maximum( [maximum( Â[i] ) for i ∈ 2:N if !isempty( Â[i] )] )
Âbins = 0:Âmax/25:Âmax

if true
    # Plot the distribution of convex hull areas for each cluster size.
    plt = plot( size=(400,200), xformatter=:plain )

    δi = round( defInt, N/5 );  δi = δi < 50 ? δi : 50
    for i ∈ δi:δi:N
        isempty( Â[i] ) && continue
        histogram!( plt, Â[i]; bins=Âbins, normalize=:probability, alpha=1/2,
            label=latexstring( "i=$(i)" ) )
    end

    plot!( plt; xlims=(0,Âmax), ylims=(0,1) )
    plot!( plt; xlabel="convex hull area", ylabel="pmf", legend=:outerright )
end

In [ ]:
ℓmax = maximum( [maximum( ℓ[i] ) for i ∈ 2:N if !isempty( ℓ[i] )] )
ℓbins = 0:ℓmax/25:ℓmax

if true
    # Plot the distribution of convex hull areas for each cluster size.
    plt = plot( size=(400,200), xformatter=:plain )

    for i ∈ δi:δi:N
        isempty( ℓ[i] ) && continue
        histogram!( plt, ℓ[i]; bins=ℓbins, normalize=:probability, alpha=1/2,
            label=latexstring( "i=$(i)" ) )
    end

    plot!( plt; xlims=(0,ℓmax), ylims=(0,1) )
    plot!( plt; xlabel="convex hull perimeter", ylabel="pmf", legend=:outerright )
end

In [ ]:
# Save results of the simulation.
if false
    # println( createfolder( folder ) )

    # Save area distribution information.
    # savedict( folder*"convex-hull-area.txt", Â )
    # savedict( folder*"convex-hull-perimeter.txt", ℓ )

    # Save scaling exponents.
    # writedlm( folder*"area-perimeter-scaling.txt", β )
    # writedlm( folder*"area-perimeter-scaling-power.txt", λ )
    # writedlm( folder*"area-perimeter-scaling-log-power.txt", γ )
end